In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(
    model=model,
    cg_method="FR"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5573282241821289
epoch:  1, loss: 0.29906898736953735
epoch:  2, loss: 0.1660996973514557
epoch:  3, loss: 0.0981469675898552
epoch:  4, loss: 0.06447339802980423
epoch:  5, loss: 0.04836566746234894
epoch:  6, loss: 0.04090287908911705
epoch:  7, loss: 0.037532225251197815
epoch:  8, loss: 0.03603638708591461
epoch:  9, loss: 0.035379618406295776
epoch:  10, loss: 0.03509151563048363
epoch:  11, loss: 0.03496352583169937
epoch:  12, loss: 0.034904468804597855
epoch:  13, loss: 0.03487506881356239
epoch:  14, loss: 0.03485829383134842
epoch:  15, loss: 0.0348220020532608
epoch:  16, loss: 0.03479035198688507
epoch:  17, loss: 0.03477305918931961
epoch:  18, loss: 0.03474409878253937
epoch:  19, loss: 0.03471016883850098
epoch:  20, loss: 0.034691859036684036
epoch:  21, loss: 0.03466937690973282
epoch:  22, loss: 0.034632228314876556
epoch:  23, loss: 0.03461286798119545
epoch:  24, loss: 0.03459751605987549
epoch:  25, loss: 0.03455726429820061
epoch:  26, loss: 0.0

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7834658622741699
Test metrics:  R2 = 0.6848494410514832


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(
    model=model,
    cg_method="PR"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.23009152710437775
epoch:  1, loss: 0.13861380517482758
epoch:  2, loss: 0.09087347984313965
epoch:  3, loss: 0.06693514436483383
epoch:  4, loss: 0.05368058383464813
epoch:  5, loss: 0.04592759907245636
epoch:  6, loss: 0.041383497416973114
epoch:  7, loss: 0.03871838375926018
epoch:  8, loss: 0.037154920399188995
epoch:  9, loss: 0.03623766824603081
epoch:  10, loss: 0.03569936752319336
epoch:  11, loss: 0.035383258014917374
epoch:  12, loss: 0.03519734740257263
epoch:  13, loss: 0.03508773073554039
epoch:  14, loss: 0.0350227952003479
epoch:  15, loss: 0.03498402610421181
epoch:  16, loss: 0.03496057540178299
epoch:  17, loss: 0.034946098923683167
epoch:  18, loss: 0.03494264557957649
epoch:  19, loss: 0.034927401691675186
epoch:  20, loss: 0.03492539003491402
epoch:  21, loss: 0.03490938991308212
epoch:  22, loss: 0.03490879386663437
epoch:  23, loss: 0.0348920114338398
epoch:  24, loss: 0.03488149493932724
epoch:  25, loss: 0.03487532213330269
epoch:  26, loss: 0

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7790132761001587
Test metrics:  R2 = 0.6996520757675171


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.ConjugateGradient(
    model=model,
    cg_method="PRP+"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.09099463373422623
epoch:  1, loss: 0.05889156833291054
epoch:  2, loss: 0.04520353302359581
epoch:  3, loss: 0.03935947269201279
epoch:  4, loss: 0.03687281161546707
epoch:  5, loss: 0.0358184315264225
epoch:  6, loss: 0.03537214174866676
epoch:  7, loss: 0.035182833671569824
epoch:  8, loss: 0.035101812332868576
epoch:  9, loss: 0.03506632149219513
epoch:  10, loss: 0.0350499302148819
epoch:  11, loss: 0.03504155948758125
epoch:  12, loss: 0.03503859043121338
epoch:  13, loss: 0.03502073138952255
epoch:  14, loss: 0.03501188009977341
epoch:  15, loss: 0.035006798803806305
epoch:  16, loss: 0.03499387949705124
epoch:  17, loss: 0.03498438745737076
epoch:  18, loss: 0.03497910872101784
epoch:  19, loss: 0.034968581050634384
epoch:  20, loss: 0.03495829924941063
epoch:  21, loss: 0.03495272248983383
epoch:  22, loss: 0.03494416922330856
epoch:  23, loss: 0.03493303433060646
epoch:  24, loss: 0.0349271260201931
epoch:  25, loss: 0.034920793026685715
epoch:  26, loss: 0.

In [10]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7736320495605469
Test metrics:  R2 = 0.6878892779350281
